# SpaNCy-Shift Stage 2 — neighbour-mean input + cross-batch MNN loss

**Question:** can Stage 2 use spatial context, or cross-batch phenotype matching, to mix batches
better **without** the marginal compression that the message-passing graph fix caused
(graph-fix: 9 shape-distorted markers, pos-pop 9/20, kBET gain inside seed noise)?

| Arm | Change vs published config (`w_adv=0.3`, α=0.6, 10 epochs) |
|---|---|
| `full` | nothing (published config) |
| `nbr` | encoder input = [own 20 markers \| mean of the 15 spatial neighbours' markers]. The neighbour mean is precomputed over the full data, so every cell sees its whole neighbourhood; the delta is still for the cell's own markers only (no averaging of expression toward neighbours). |
| `mnn` | + cross-batch MNN loss (`w_mnn=0.5`, k=5): mutual nearest neighbours across batches in Stage 1 expression are pulled together on unimodal markers |
| `nbr+mnn` | both |

Every arm in a seed starts from the same init (`torch.manual_seed(seed)` before each `train()`).
Both additions are cheap: the neighbour mean is one precomputed (N, 20) array, and MNN is a
≤512×512 distance matrix per batch pair per step. Training time per arm is printed so you can check.

**Pre-set rule (§9):** an arm is **ADOPTED** only if, paired against `full` within seed, kBET is
higher at **every** seed AND silhouette is ≥ `full` − 0.005 at every seed AND it adds no
shape-distorted markers on average AND bimodal markers stay untouched (`bim_delta` ≤ 0.001).
Seed noise is ~0.016 kBET, so a small sign-consistent gain is suggestive, not proof.

**To run:** Runtime → Run all (GPU). Upload **`spancy_shift_nbrmean.py`** when §0 asks.
Runtime ≈ 12 trainings × ~8 min ≈ 1.6 h + kBET on 12 layers. Quick pass: `SEEDS = [0]` in §3.
**Paste back §9** (plus any error).

## 0. Colab Setup

In [ ]:
# Install torch-geometric (auto-detect Colab's torch + CUDA version)
import torch
tv = torch.__version__.split('+')[0]
cv = torch.version.cuda or 'cpu'
if cv != 'cpu':
    cv = 'cu' + cv.replace('.', '')
whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
print(f'PyTorch {tv}, CUDA tag {cv}')
!pip install -q torch-scatter torch-sparse torch-cluster torch-geometric -f {whl}
!pip install -q "pandas<3" "anndata<0.11" scanpy pegasuspy pegasusio
print('Done.')

In [ ]:
# Upload spancy_shift_nbrmean.py (requested only if missing).
from google.colab import files
import os

NEEDED = ['spancy_shift_nbrmean.py']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print(f'Please upload: {missing}')
    files.upload()

for f in NEEDED:
    assert os.path.exists(f), f'{f} not found — upload it (Colab renames re-uploads to "name (1).py")'
    print(f'{f:24s} {os.path.getsize(f):>10,} bytes')

In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import matplotlib.pyplot as plt
from scipy.stats import iqr as _iqr
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture

import spancy_shift_nbrmean as ss
from spancy_shift_nbrmean import (
    load_adata, detect_bimodal_markers, find_best_sample_per_marker,
    positive_population_table,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
!wget -q https://zenodo.org/records/16383766/files/PRAD_anndata.h5ad
print('Download complete.')

## 1. Load data, bimodal markers, reference samples

In [ ]:
DATA_PATH = 'PRAD_anndata.h5ad'

adata = load_adata(DATA_PATH)
print(adata)
print(f"\nMarkers ({adata.n_vars}): {list(adata.var_names)}")
print(f"Batches:  {sorted(adata.obs['batch_id'].unique().tolist())}")
print(f"Samples:  {sorted(adata.obs['sample_id'].unique().tolist())}")

marker_names = list(adata.var_names)
X_raw = np.asarray(adata.X.toarray() if sp.issparse(adata.X) else adata.X)
batch_vals = adata.obs['batch_id'].values
unique_batches = sorted(adata.obs['batch_id'].unique())

In [ ]:
BIMODAL_MIN_BATCH_FRAC = 0.5

X_log_preview = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)
batch_codes_preview = adata.obs['batch_id'].astype('category').cat.codes.values.astype('int64')

marker_is_bimodal, thresholds = detect_bimodal_markers(
    X_log_preview, marker_names,
    batch_codes=batch_codes_preview,
    min_prominence_frac=0.05,
    bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
)

print(f"Bimodal markers ({marker_is_bimodal.sum()}):")
for i, name in enumerate(marker_names):
    if marker_is_bimodal[i]:
        print(f"  {name:>12s}  threshold={thresholds[i]:.3f}")
print(f"\nUnimodal markers ({(~marker_is_bimodal).sum()}):")
for i, name in enumerate(marker_names):
    if not marker_is_bimodal[i]:
        print(f"  {name}")
print("\nNote: DDPM scores the full joint distribution — no special handling required.")
bimodal_names = [marker_names[i] for i in range(len(marker_names)) if marker_is_bimodal[i]]

In [ ]:
ref_sample_per_marker = find_best_sample_per_marker(adata)
uni_markers = [m for m in marker_names if m not in bimodal_names]
uni_idx = [marker_names.index(m) for m in uni_markers]
bim_idx = [marker_names.index(m) for m in bimodal_names]
print(f'{len(uni_markers)} unimodal markers; bimodal: {bimodal_names}')

## 3. Train the arms

Per seed, four trainings from the same init. Inference uses a fixed seed. Training minutes and the
average number of MNN pairs per step are recorded for §9.

In [ ]:
SEEDS = [0, 1, 2]     # [0] = quick pass (~30 min of training)
KBET_SEEDS = SEEDS    # kBET IS a target here -> all seeds (noise ~0.016)
POP_SEEDS = [0]
K_NEIGHBORS = 15

BASE = dict(
    n_epochs=10, lr=1e-3, warmup_epochs=2, n_per_batch=512, k_neighbors=K_NEIGHBORS,
    hybrid_alpha=0.6, temperature=0.07, w_recon=0.1, w_contrast=0.5, w_adv=0.3, w_mmd=1.0,
    mmd_samples=256, grl_max=1.0, use_nbr_mean=False, w_mnn=0.0, mnn_k=5,
)
W_MNN = 0.5
ARMS = {
    'full':    {},
    'nbr':     {'use_nbr_mean': True},
    'mnn':     {'w_mnn': W_MNN},
    'nbr+mnn': {'use_nbr_mean': True, 'w_mnn': W_MNN},
}
KEY = {'full': 'full', 'nbr': 'nbr', 'mnn': 'mnn', 'nbr+mnn': 'nbrmnn'}


def lkey(arm, seed):
    return f'{KEY[arm]}_s{seed}'


adata_norm = None
train_info = {}
t0 = time.time()
for seed in SEEDS:
    for arm, over in ARMS.items():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        t_arm = time.time()
        m, sc, ref, hist = ss.train(
            adata, device_str=DEVICE,
            bimodal_min_batch_frac=BIMODAL_MIN_BATCH_FRAC,
            ref_sample_per_marker=ref_sample_per_marker,
            **{**BASE, **over},
        )
        train_min = (time.time() - t_arm) / 60
        torch.manual_seed(100)
        torch.cuda.manual_seed_all(100)
        out = ss.normalize_adata(
            adata, m, sc, ref, hybrid_alpha=BASE['hybrid_alpha'], k_neighbors=K_NEIGHBORS,
            inference_batch_size=2048, device_str=DEVICE,
            layer_name='normalized', keep_base_layer=True,
        )
        g = np.asarray(out.layers['normalized'], dtype=np.float32)
        if adata_norm is None:
            adata_norm = out                  # keeps 'normalized_base' (Stage 1)
            del adata_norm.layers['normalized']
            X_base = np.asarray(adata_norm.layers['normalized_base'])
            base_log = np.log1p(np.clip(X_base, 0, None)).astype(np.float32)
        else:
            del out
        adata_norm.layers[lkey(arm, seed)] = g
        train_info[lkey(arm, seed)] = dict(train_min=train_min,
                                           mnn_pairs=float(np.mean(hist['mnn_pairs'])))
        del g, m, sc, ref, hist
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'  seed {seed}  {arm:8s} done  train {train_min:.1f} min  '
              f'mnn pairs/step {train_info[lkey(arm, seed)]["mnn_pairs"]:.0f}  '
              f'(total {(time.time() - t0) / 60:.1f} min)')

LAYERS = [('normalized_base', 'Stage 1', None, None)]
LAYERS += [(lkey(a, s), f'{a} s{s}', a, s) for s in SEEDS for a in ARMS]
print(f'\n{len(LAYERS)} layers: {[l[1] for l in LAYERS]}')

In [ ]:
# 3b. Divergence check — run after §3, before §4.
# Without the Huber term (no-Huber, all-off) nothing bounds the size of the correction, and the
# expm1 at inference overflows (max 4e10 / inf). Such layers break the metric cells (inf -> histogram
# and silhouette errors), so they are reported here and dropped from LAYERS: their verdict is
# "diverged". Criterion: any non-finite value, or > 0.1% of entries above 1.2x the raw per-marker
# log1p max (the same 1.2x cap rule as SpaNCy-GNN).
CAP = 1.2
raw_max = np.log1p(np.clip(X_raw, 0, None).max(axis=0))
diverged, div_rows = {}, []
for key, label, arm, seed in LAYERS:
    X = np.asarray(adata_norm.layers[key])
    with np.errstate(over='ignore', invalid='ignore'):
        X_log = np.log1p(np.clip(X, 0, None))
    finite = np.isfinite(X_log)
    over = ~finite | (X_log > CAP * raw_max)
    frac_over = over.mean(axis=0)
    bad_markers = [f'{marker_names[i]} {100 * frac_over[i]:.1f}%'
                   for i in np.argsort(-frac_over) if frac_over[i] > 0.001]
    is_div = (~finite).any() or over.mean() > 0.001
    if is_div and arm is not None:
        diverged[key] = (arm, seed)
    div_rows.append(dict(layer=label, nonfinite=int((~finite).sum()),
                         pct_over_cap=round(100 * float(over.mean()), 3),
                         max_log1p=round(float(X_log[finite].max()), 2),
                         diverged=is_div, markers_over_cap=', '.join(bad_markers[:6])))
    del X, X_log, finite, over
    gc.collect()

print(f'Raw log1p max over markers: {raw_max.max():.2f}  (cap = {CAP}x per marker)')
print(pd.DataFrame(div_rows).to_string(index=False))
LAYERS = [l for l in LAYERS if l[0] not in diverged]
print(f'\nDropped {len(diverged)} diverged layers from the metric cells: '
      f'{sorted(f"{a} s{s}" for a, s in diverged.values())}')

## 4. 1D shape + correction on bimodal markers

Same per-(marker, sample) ratios as cell 6b of the other notebooks (`_shape_ratios` copied verbatim),
but computed for **all** markers so the bimodal ones (ECAD, …) are visible. `bim_delta` = mean
|log1p(arm) − log1p(Stage 1)| over bimodal markers: ~0 means bimodal markers were left alone.

In [ ]:
N_SHAPE_BINS = 80
MIN_CELLS = 50
PEAK_LO, VAR_LO, VAR_HI = 0.80, 0.80, 1.25

sample_ids_shape = adata_norm.obs['sample_id'].values
sample_idx = {s: np.where(sample_ids_shape == s)[0] for s in np.unique(sample_ids_shape)}
sample_idx = {s: idx for s, idx in sample_idx.items() if idx.size >= MIN_CELLS}
X_raw_log = np.log1p(np.clip(X_raw, 0, None)).astype(np.float32)


def _shape_ratios(raw_vals, norm_vals):
    lo = min(raw_vals.min(), norm_vals.min())
    hi = max(raw_vals.max(), norm_vals.max())
    if not np.isfinite(lo) or hi <= lo:
        return np.nan, np.nan, np.nan
    edges = np.linspace(lo, hi, N_SHAPE_BINS + 1)
    hr, _ = np.histogram(raw_vals,  bins=edges, density=True)
    hn, _ = np.histogram(norm_vals, bins=edges, density=True)
    peak = hn.max() / hr.max() if hr.max() > 0 else np.nan
    vr   = norm_vals.var() / raw_vals.var() if raw_vals.var() > 0 else np.nan
    ir_r = _iqr(raw_vals)
    ir   = _iqr(norm_vals) / ir_r if ir_r > 0 else np.nan
    return peak, vr, ir


shape_rows, shape_sum = [], {}
for key, label, arm, seed in LAYERS:
    X_log = np.log1p(np.clip(np.asarray(adata_norm.layers[key]), 0, None)).astype(np.float32)
    for mi, mname in enumerate(marker_names):
        ps, vs, irs = [], [], []
        for idx in sample_idx.values():
            p, v, ir = _shape_ratios(X_raw_log[idx, mi], X_log[idx, mi])
            if np.isfinite(p):  ps.append(p)
            if np.isfinite(v):  vs.append(v)
            if np.isfinite(ir): irs.append(ir)
        shape_rows.append(dict(key=key, marker=mname, peak=np.mean(ps), var=np.mean(vs), iqr=np.mean(irs)))
    df = pd.DataFrame([r for r in shape_rows if r['key'] == key]).set_index('marker')
    bad = df[(df.peak < PEAK_LO) | (df['var'] < VAR_LO) | (df['var'] > VAR_HI)]
    u = df.loc[uni_markers]
    shape_sum[key] = dict(
        peak=float(u.peak.mean()), var=float(u['var'].mean()), iqr=float(u.iqr.mean()),
        dist=len(bad), dist_list=', '.join(bad.index),
        ecad_var=float(df.loc['ECAD', 'var']) if 'ECAD' in df.index else np.nan,
        ecad_iqr=float(df.loc['ECAD', 'iqr']) if 'ECAD' in df.index else np.nan,
        bim_delta=float(np.abs(X_log[:, bim_idx] - base_log[:, bim_idx]).mean()) if bim_idx else 0.0,
        uni_delta=float(np.abs(X_log[:, uni_idx] - base_log[:, uni_idx]).mean()),
    )
    del X_log
    gc.collect()
shape_df = pd.DataFrame(shape_rows)

print('Shape summary (peak/var/iqr = unimodal means; dist = ALL markers incl. bimodal):')
print(pd.DataFrame(shape_sum).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).round(4).to_string())

## 5. Silhouette (all seeds)

In [ ]:
# Silhouette — UniFORM-style, 3 cell types, PER SAMPLE (same method + guard as the other notebooks).
N_SIL = 3000
MIN_TYPE_CELLS = 50

anchor_thresholds = {}
for mname in ['ECAD', 'CD45', 'aSMA']:
    mi = marker_names.index(mname)
    gmm = GaussianMixture(n_components=2, random_state=0).fit(X_log_preview[:, mi].reshape(-1, 1))
    anchor_thresholds[mname] = float(np.mean(gmm.means_))
print('Anchor GMM thresholds (log1p, raw):', {k: round(v, 3) for k, v in anchor_thresholds.items()})

cell_type = np.full(len(X_log_preview), -1, dtype=np.int8)
cell_type[X_log_preview[:, marker_names.index('aSMA')] > anchor_thresholds['aSMA']] = 2
cell_type[X_log_preview[:, marker_names.index('CD45')] > anchor_thresholds['CD45']] = 1
cell_type[X_log_preview[:, marker_names.index('ECAD')] > anchor_thresholds['ECAD']] = 0

# Per-sample selection is decided once (deterministic guard, rng 42 per sample as before).
sample_ids_all = adata_norm.obs['sample_id'].values
sil_sel, dropped = {}, []
for sample in sorted(np.unique(sample_ids_all)):
    smask_idx = np.where(sample_ids_all == sample)[0]
    s_labels = cell_type[smask_idx]
    big = [c for c in (0, 1, 2) if int((s_labels == c).sum()) >= MIN_TYPE_CELLS]
    if len(big) < 2:
        dropped.append(sample)
        continue
    valid = np.where(np.isin(s_labels, big))[0]
    rng = np.random.default_rng(42)
    sel = valid[rng.choice(len(valid), min(N_SIL, len(valid)), replace=False)]
    sil_sel[sample] = (smask_idx[sel], s_labels[sel])

sil_rows = {}
for key in ['raw'] + [k for k, _, _, _ in LAYERS]:
    X = X_raw if key == 'raw' else np.asarray(adata_norm.layers[key])
    sil_rows[key] = {s: float(silhouette_score(np.log1p(np.clip(X[ii], 0, None)), lab, metric='euclidean'))
                     for s, (ii, lab) in sil_sel.items()}
sil_df = pd.DataFrame(sil_rows)
sil_df.loc['Mean'] = sil_df.mean()
print(f'{len(sil_sel)} samples passed the guard (dropped: {dropped})')
print(sil_df.loc[['Mean']].round(4).T.rename(index=dict((k, l) for k, l, _, _ in LAYERS)).to_string())

## 6. Positive population (`POP_SEEDS`)

In [ ]:
pop_sum = {}
for key, label, arm, seed in LAYERS:
    if seed is not None and seed not in POP_SEEDS:
        continue
    pt = positive_population_table(adata_norm, norm_layer=key, sample_col='sample_id')
    s = pt.groupby('marker')['delta'].mean()
    pop_sum[key] = dict(pass_5pct=int((s.abs() < 5).sum()),
                        mean_abs_delta=float(pt['delta'].abs().mean()))
    print(f'  {label:16s} {pop_sum[key]["pass_5pct"]}/20 within +/-5%   '
          f'mean |delta| {pop_sum[key]["mean_abs_delta"]:.2f}%')

## 8. kBET (`KBET_SEEDS`)

In [ ]:
import pegasus as pg
import pegasusio as pgio

def subset_ab(adata, sample, batch):
    mask = (adata.obs['sample_id'] == sample) & (adata.obs['batch_id'] == batch)
    return adata[mask, :].copy()

GROUP_DEFS = {
    'g1': [('PRAD-02', 'batch1'), ('PRAD-14', 'batch7')],
    'g2': [('PRAD-01', 'batch4'), ('PRAD-02', 'batch1')],
    'g3': [('PRAD-05', 'batch1'), ('PRAD-12', 'batch2')],
    'g4': [('PRAD-07', 'batch2'), ('PRAD-19', 'batch6')],
    'g5': [('PRAD-02', 'batch1'), ('PRAD-07', 'batch2')],
}

# Stage 1 kBET is deterministic (0.6202) -> skipped.
ABL_LABEL = {k: l for k, l, _, _ in LAYERS}
EVAL_LAYERS = [k for k, _, _, s in LAYERS if s is not None and s in KBET_SEEDS]
print(f'kBET on {len(EVAL_LAYERS)} layers: {[ABL_LABEL[k] for k in EVAL_LAYERS]}')

In [ ]:
import anndata as ad
import gc

all_kbet = {}
umap_coords = {}

# MEMORY NOTE: the original cell did `adata_kbet = adata_norm.copy()`, which duplicates
# EVERY layer (~141 MB each; 6 layers once the ablation arms exist = ~850 MB per
# iteration, 5 iterations). pegasus only ever reads .X here, so build a minimal
# AnnData instead: X + obs + var. That keeps the per-iteration cost at one 141 MB
# array rather than ~850 MB, and it is faster too.
for layer_name in EVAL_LAYERS:
    label = ABL_LABEL.get(layer_name, layer_name)
    print(f'\n{"-"*55}\n  {label}  ({layer_name})\n{"-"*55}')

    adata_kbet = ad.AnnData(
        X=np.asarray(adata_norm.layers[layer_name], dtype=np.float32).copy(),
        obs=adata_norm.obs.copy(),
        var=adata_norm.var.copy(),
    )

    layer_res = {}
    for gname, pairs in GROUP_DEFS.items():
        parts = adata_g = mmdata = None
        try:
            parts = [subset_ab(adata_kbet, s, b) for s, b in pairs]
            adata_g = parts[0].concatenate(parts[1], batch_key=None)
            mmdata = pgio.MultimodalData(adata_g)
            pg.qc_metrics(mmdata)
            pg.filter_data(mmdata)
            pg.identify_robust_genes(mmdata)
            pg.pca(mmdata, features=None, n_components=20)
            pg.neighbors(mmdata)
            pg.umap(mmdata)
            umap_coords.setdefault(layer_name, {})[gname] = (
                mmdata.obsm['X_umap'].copy(),
                mmdata.obs['batch_id'].values.copy(),
            )
            stat, pval, score = pg.calc_kBET(mmdata, attr='batch_id', rep='umap')
            layer_res[gname] = {'kBET': score, 'chi2': stat, 'p': pval}
            print(f'  {gname}: kBET={score:.4f}  chi2={stat:.4f}  p={pval:.4f}')
        except Exception as e:
            print(f'  {gname}: FAILED — {e}')
            layer_res[gname] = {'kBET': float('nan'), 'chi2': float('nan'), 'p': float('nan')}
        finally:
            del parts, adata_g, mmdata

    all_kbet[layer_name] = layer_res
    del adata_kbet
    gc.collect()

print('\nDone.')

## 9. Result (paste this)

Paired within seed against `full`. Pre-set rule: **ADOPT** only if kBET is higher at every seed,
silhouette ≥ `full` − 0.005 at every seed, mean shape-distorted markers not increased, and
`bim_delta` ≤ 0.001. The numbers are printed either way.

In [ ]:
recs = []
for key, label, arm, seed in LAYERS:
    if arm is None:
        continue
    r = dict(arm=arm, seed=seed, sil=float(sil_df.loc['Mean', key]), **train_info[key], **{
        k: v for k, v in shape_sum[key].items() if k != 'dist_list'})
    if key in all_kbet:
        kv = [x['kBET'] for x in all_kbet[key].values() if not np.isnan(x['kBET'])]
        r['kBET'] = float(np.mean(kv)) if kv else np.nan
    if key in pop_sum:
        r['pos_pop'] = pop_sum[key]['pass_5pct']
    recs.append(r)
res = pd.DataFrame(recs)

cols = [c for c in ['kBET', 'sil', 'peak', 'var', 'iqr', 'dist', 'ecad_var', 'bim_delta',
                    'uni_delta', 'train_min', 'mnn_pairs'] if c in res]
agg = res.groupby('arm', sort=False)[cols].agg(['mean', 'std']).round(4)
s1 = shape_sum['normalized_base']
print('=' * 100)
print(f'MEAN +/- SD over seeds {SEEDS}  (unimodal peak/var/iqr; dist = all markers; '
      f'bim_delta / uni_delta = mean |correction| in log1p)')
print('=' * 100)
print(agg.to_string())
print(f'\nStage 1:  kBET 0.6202 (canonical)  sil {float(sil_df.loc["Mean", "normalized_base"]):.4f}  '
      f'peak {s1["peak"]:.3f} var {s1["var"]:.3f} iqr {s1["iqr"]:.3f}  dist {s1["dist"]}   '
      f'raw sil {float(sil_df.loc["Mean", "raw"]):.4f}')
print('Reference: published full = kBET 0.709 +/- 0.006, sil 0.360 +/- 0.002; '
      'graph-fix = 9 distorted, pos-pop 9/20')

print('\nDivergence (§3b):')
print(pd.DataFrame(div_rows).to_string(index=False))

print('\nPer seed:')
print(res[['arm', 'seed'] + [c for c in ['kBET', 'sil', 'dist', 'pos_pop', 'train_min'] if c in res]]
      .round(4).to_string(index=False))

print('\nPer-group kBET:')
for key, label, arm, seed in LAYERS:
    if key in all_kbet:
        print(f'  {label:14s} ' + '  '.join(f'{g} {v["kBET"]:.3f}' for g, v in all_kbet[key].items()))

print('\nDistorted markers, seed 0:')
for key, label, arm, seed in LAYERS:
    if seed == SEEDS[0]:
        print(f'  {label:14s} {shape_sum[key]["dist"]}  {shape_sum[key]["dist_list"]}')

print('\n' + '=' * 100)
print('PER ARM — vs full, paired within seed')
print('=' * 100)
full = res[res.arm == 'full'].set_index('seed')
for arm in [a for a in ARMS if a != 'full']:
    n_div = sum(1 for k in diverged if k.startswith(KEY[arm] + '_s'))
    a = res[res.arm == arm].set_index('seed')
    if a.empty:
        print(f'  {arm:8s} DIVERGED at all {n_div} seeds (see §3b) -> REJECT')
        continue
    seeds = [s for s in a.index if s in full.index]
    dk = (a.loc[seeds, 'kBET'] - full.loc[seeds, 'kBET']) if 'kBET' in res else pd.Series(dtype=float)
    ds = a.loc[seeds, 'sil'] - full.loc[seeds, 'sil']
    ddist = float((a.loc[seeds, 'dist'] - full.loc[seeds, 'dist']).mean())
    bim = float(a['bim_delta'].max())
    checks = {
        'kBET higher every seed': bool(len(dk) == len(seeds) and dk.notna().all() and (dk > 0).all()),
        'silhouette >= full - 0.005 every seed': bool((ds >= -0.005).all()),
        'no extra distorted markers': ddist <= 0,
        'bimodal untouched': bim <= 0.001,
        'no divergence': n_div == 0,
    }
    verdict = 'ADOPT' if all(checks.values()) else 'REJECT'
    print(f'  {arm:8s} kBET ' + ' '.join(f's{s} {v:+.4f}' for s, v in dk.items())
          + f' (mean {dk.mean():+.4f})'
          + ' | sil ' + ' '.join(f's{s} {v:+.4f}' for s, v in ds.items())
          + f' | dist {ddist:+.1f} | bim_delta max {bim:.4f}')
    for name, ok in checks.items():
        print(f'            [{"x" if ok else " "}] {name}')
    print(f'            -> {verdict}')